In [9]:
import json
import os
from collections import defaultdict

class AVLNode:
    """Nodo del árbol AVL que almacena preguntas y respuestas"""
    def __init__(self, data):
        self.data = data
        self.left = None
        self.right = None
        self.height = 1
        self.questions_by_subject = {}

    def update_height(self):
        left_height = self.left.height if self.left else 0
        right_height = self.right.height if self.right else 0
        self.height = 1 + max(left_height, right_height)

    def balance_factor(self):
        left_height = self.left.height if self.left else 0
        right_height = self.right.height if self.right else 0
        return left_height - right_height


class AVLTree:
    """Árbol AVL para almacenar y organizar preguntas del cuestionario"""
    def __init__(self):
        self.root = None
        self.size = 0

    def insert(self, data):
        if not isinstance(data, dict) or 'id' not in data:
            raise ValueError("Datos deben contener 'id'")
        self.root = self._insert(self.root, data)
        self.size += 1

    def _insert(self, node, data):
        if not node:
            new_node = AVLNode(data)
            self._add_to_subject_index(new_node)
            return new_node

        if data['id'] < node.data['id']:
            node.left = self._insert(node.left, data)
        else:
            node.right = self._insert(node.right, data)

        node.update_height()
        return self._balance(node)

    def _balance(self, node):
        balance = node.balance_factor()

        if balance > 1:
            if node.left.balance_factor() >= 0:
                return self._right_rotate(node)
            else:
                node.left = self._left_rotate(node.left)
                return self._right_rotate(node)
        elif balance < -1:
            if node.right.balance_factor() <= 0:
                return self._left_rotate(node)
            else:
                node.right = self._right_rotate(node.right)
                return self._left_rotate(node)

        return node

    def _left_rotate(self, z):
        y = z.right
        T2 = y.left
        y.left = z
        z.right = T2
        z.update_height()
        y.update_height()
        return y

    def _right_rotate(self, z):
        y = z.left
        T3 = y.right
        y.right = z
        z.left = T3
        z.update_height()
        y.update_height()
        return y

    def _add_to_subject_index(self, node):
        subject = node.data.get('subject', 'general')
        if subject not in node.questions_by_subject:
            node.questions_by_subject[subject] = []
        node.questions_by_subject[subject].append(node.data)

    def search_by_subject(self, subject):
        questions = []
        self._collect_by_subject(self.root, subject, questions)
        return questions

    def _collect_by_subject(self, node, subject, questions):
        if not node:
            return
        if subject in node.questions_by_subject:
            questions.extend(node.questions_by_subject[subject])
        self._collect_by_subject(node.left, subject, questions)
        self._collect_by_subject(node.right, subject, questions)

    def get_all_subjects(self):
        subjects = set()
        self._collect_subjects(self.root, subjects)
        return sorted(subjects)


class QuestionParser:
    """Clase para procesar el archivo JSON de preguntas"""
    @staticmethod
    def parse_questions(json_data):
        questions = []
        current_question = None
        question_id = 1
        
        for line in json_data['lineas']:
            line = line.strip()
            
            if line.startswith("¿"):
                if current_question:
                    questions.append(current_question)
                    question_id += 1
                
                current_question = {
                    'id': question_id,
                    'question': line,
                    'options': [],
                    'answer': '',
                    'subject': 'psicologia',
                    'bibliography': ''
                }
            elif line.startswith(('a.', 'b.', 'c.')):
                if current_question:
                    option_text = line[2:].split('Bibliografía:')[0].strip()
                    current_question['options'].append(option_text)
                    
                    if 'Bibliografía:' in line:
                        current_question['bibliography'] = line.split('Bibliografía:')[-1].strip()
            elif line.startswith("La respuesta correcta es:"):
                if current_question:
                    current_question['answer'] = line.replace("La respuesta correcta es:", "").strip()
        
        if current_question:
            questions.append(current_question)
        
        return questions


class SubjectClassifier:
    """Clasifica preguntas por materia basado en palabras clave"""
    def __init__(self):
        self.keywords = {
            'mindfulness': ['mindfulness', 'meditación', 'respiración'],
            'ansiedad': ['ansiedad', 'preocupación', 'temor'],
            'estrés': ['estrés', 'presión', 'tensión'],
            'relajación': ['relajación', 'calma', 'tranquilidad'],
            'psicologia': ['psicológico', 'mental', 'emocional']
        }

    def classify(self, text):
        text = text.lower()
        scores = defaultdict(int)
        
        for subject, words in self.keywords.items():
            for word in words:
                if word in text:
                    scores[subject] += 1
        
        return max(scores.items(), key=lambda x: x[1])[0] if scores else 'psicologia'


def load_questions_from_json(filepath):
    """Carga y procesa preguntas desde el archivo JSON"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
    except FileNotFoundError:
        # Si no encuentra el archivo, usar datos embebidos
        print(f"Advertencia: No se encontró el archivo {filepath}, usando datos de ejemplo")
        json_data = {
            "lineas": [
                "¿Qué se recomienda hacer si la mente se distrae durante la práctica de mindfulness?",
                "a. Terminar la sesión de meditación",
                "b. Redirigir gentilmente la atención a la respiración",
                "c. Ignorar la distracción",
                "La respuesta correcta es: Redirigir gentilmente la atención a la respiración",
                "¿Cómo se define la ansiedad según los expertos?",
                "a. Como una respuesta emocional normal",
                "b. Como un trastorno psicológico complejo",
                "c. Como una respuesta emocional caracterizada por preocupación excesiva",
                "La respuesta correcta es: Como una respuesta emocional caracterizada por preocupación excesiva"
            ]
        }
    
    # Parsear preguntas del formato especial
    parser = QuestionParser()
    questions = parser.parse_questions(json_data)
    
    # Clasificar por materia
    classifier = SubjectClassifier()
    for q in questions:
        q['subject'] = classifier.classify(q['question'])
    
    return questions


def main():
    # Obtener la ruta absoluta del directorio actual
    current_dir = os.path.dirname(os.path.abspath(__file__))
    json_path = os.path.join(current_dir, 'bancodepreguntas_global_cienciadatos.json')
    
    # Cargar preguntas desde el JSON
    questions = load_questions_from_json(json_path)
    
    # Crear y poblar el árbol AVL
    avl_tree = AVLTree()
    for q in questions:
        avl_tree.insert(q)
    
    # Mostrar información básica
    print("=== CUESTIONARIO DE PSICOLOGÍA ===")
    print(f"Total preguntas cargadas: {avl_tree.size}")
    
    print("\nMaterias disponibles:")
    for subject in avl_tree.get_all_subjects():
        print(f"- {subject.capitalize()}")
    
    # Mostrar algunas preguntas como ejemplo
    print("\nEjemplo de preguntas:")
    sample_questions = avl_tree.search_by_subject('mindfulness')[:2] + avl_tree.search_by_subject('ansiedad')[:2]
    for q in sample_questions:
        print(f"\nID: {q['id']}")
        print(f"Pregunta: {q['question']}")
        print("Opciones:")
        for i, opt in enumerate(q['options'], 1):
            print(f"  {i}. {opt}")
        print(f"Respuesta correcta: {q['answer']}")
        if q['bibliography']:
            print(f"Fuente: {q['bibliography']}")

if __name__ == '__main__':
    main()

NameError: name '__file__' is not defined